In [1]:
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [5]:
class CarlaDataset(Dataset):

    def __init__(self, csv_path, image_dir, label_column):

        self.data = pd.read_csv(csv_path)

        self.image_dir = image_dir

        self.label_column = label_column

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        frame = row["frame"]

        image_name = f"{int(frame):06d}.jpg"

        image_path = f"{self.image_dir}/{image_name}"

        image = Image.open(image_path).convert("RGB")

        image = self.transform(image)

        label = int(row[self.label_column])

        return image, label

In [11]:
path = "E://OVGU_NOTES//Machine_Learning_Safety//assignment//Data_set//assigment3//test"
test_dataset = CarlaDataset(
    csv_path=f"{path}//labels.csv",
    image_dir=f"{path}//rgb-front",
    label_column="has_pedestrian"
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)


In [13]:
# ----------------------------
# MODEL
# ----------------------------
mode_path = "E://OVGU_NOTES//Machine_Learning_Safety//assignment"
model = models.resnet18(weights=None)

model.fc = nn.Linear(model.fc.in_features, 1)

model.load_state_dict(
    torch.load(f"{mode_path}//pedestrian_vehicle_Traffic.pth")
)

model = model.to(device)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [14]:
all_logits = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        all_logits.extend(outputs.cpu().numpy().flatten())

        all_labels.extend(labels.numpy())

In [15]:
# ----------------------------
# TEMPERATURE SCALING
# ----------------------------

# Safety constraint threshold from Exercise 5.4:
# "If model confidence is below theta=0.6, reduce speed to <= 15 km/h"

TEMPERATURES = [0.5, 1.0, 2.0]

DECISION_THRESHOLD = 0.5

SAFETY_THETA = 0.6

In [16]:
import torch as _torch

logits_tensor = _torch.tensor(all_logits)

print("\n" + "=" * 60)
print("TEMPERATURE SCALING RESULTS")
print("Decision threshold : 0.5")
print("Safety theta       : 0.6")
print("=" * 60)

for T in TEMPERATURES:

    # Apply temperature scaling: p_T = sigmoid(z / T)
    scaled_logits = logits_tensor / T

    probs = _torch.sigmoid(scaled_logits).numpy()

    preds = (probs > DECISION_THRESHOLD).astype(int)

    accuracy  = accuracy_score(all_labels, preds)
    precision = precision_score(all_labels, preds, zero_division=0)
    recall    = recall_score(all_labels, preds, zero_division=0)
    f1        = f1_score(all_labels, preds, zero_division=0)

    # How many predictions fall below the safety confidence threshold?
    below_theta = (probs < SAFETY_THETA).sum()
    below_theta_pct = below_theta / len(probs) * 100

    print(f"\n--- Temperature T = {T} ---")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Predictions below safety theta (< {SAFETY_THETA}): "
          f"{below_theta} / {len(probs)}  ({below_theta_pct:.1f}%)")
    print(f"  -> Safety constraint triggers for {below_theta_pct:.1f}% of images")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("T=0.5 : Sharpens probabilities toward 0 or 1 (overconfident).")
print("        Fewer images trigger the safety speed constraint.")
print("        LESS SAFE — model is confident even when wrong.")
print()
print("T=1.0 : No scaling applied. Baseline model behaviour.")
print()
print("T=2.0 : Flattens probabilities toward 0.5 (underconfident).")
print("        More images fall below theta and trigger speed reduction.")
print("        MORE CONSERVATIVE — but may over-trigger the constraint.")
print()
print("Accuracy alone is NOT sufficient to verify the safety constraint.")
print("You must also measure CALIBRATION: whether the model's confidence")
print("actually reflects its true probability of being correct.")


TEMPERATURE SCALING RESULTS
Decision threshold : 0.5
Safety theta       : 0.6

--- Temperature T = 0.5 ---
Accuracy  : 0.5408
Precision : 0.2180
Recall    : 0.5184
F1 Score  : 0.3069
Predictions below safety theta (< 0.6): 2156 / 3600  (59.9%)
  -> Safety constraint triggers for 59.9% of images

--- Temperature T = 1.0 ---
Accuracy  : 0.5408
Precision : 0.2180
Recall    : 0.5184
F1 Score  : 0.3069
Predictions below safety theta (< 0.6): 2398 / 3600  (66.6%)
  -> Safety constraint triggers for 66.6% of images

--- Temperature T = 2.0 ---
Accuracy  : 0.5408
Precision : 0.2180
Recall    : 0.5184
F1 Score  : 0.3069
Predictions below safety theta (< 0.6): 2812 / 3600  (78.1%)
  -> Safety constraint triggers for 78.1% of images

INTERPRETATION
T=0.5 : Sharpens probabilities toward 0 or 1 (overconfident).
        Fewer images trigger the safety speed constraint.
        LESS SAFE — model is confident even when wrong.

T=1.0 : No scaling applied. Baseline model behaviour.

T=2.0 : Flattens pr